In [ ]:
import uproot
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import mplhep

In [ ]:
events = uproot.open("/afs/cern.ch/work/a/amascell/FASERCal/FASERCal_converter/build/output_32ch-1C0-longRun_daq.root:Events").arrays()
feb_vars = ['feb_id', 'channel_id', 'amplitude_lg', 'amplitude_hg', 
            'hit_time_rise', 'hit_time_fall', 'gts_time_rise', 'gts_time_fall']

In [ ]:
np.unique(ak.flatten(events['feb_id']))
len(events)

In [ ]:
def chunk_list(lst, size):
    return [lst[i:i + size] for i in range(0, len(lst), size)]

### Plot Amplitude Distributions

In [ ]:
mplhep.style.use("CMS")

# var : [hist_bins, hist_range]
amp_var_dict = {
    'amplitude_lg': [200, [0, 200]],
    'amplitude_hg': [600, [0, 600]]
}

for feb_id in np.unique(ak.flatten(events['feb_id'])):
    feb_events = events[feb_vars]
    feb_events = feb_events[feb_events['feb_id'] == feb_id]
    
    for amp_var in amp_var_dict:
        hist_range = amp_var_dict[amp_var][1]
        hist_bins = amp_var_dict[amp_var][0]
        
        for ch_list in chunk_list(list(np.unique(ak.flatten(feb_events['channel_id']))), 8):
            fig = plt.subplots(figsize=[10, 7])
            for ch in ch_list:
                amplitude_ch = feb_events[amp_var][(feb_events['channel_id'] == ch) & ((feb_events[amp_var] != -1))]
                assert ~(ak.any(ak.num(amplitude_ch, axis=-1) > 1)), f"More than 1 amplitude value for channel {ch}"
                plt.hist(ak.flatten(amplitude_ch, axis=-1), histtype='step', linewidth=1.5, label=f"Channel {ch}", range=hist_range, bins=hist_bins)

            plt.legend(loc="upper right", fontsize=20)
            plt.xlabel(amp_var)
            plt.ylabel("Entries")
            plt.yscale("log")
            plt.show()
            plt.close()

### Study missing LG and HG values

In [ ]:
fig = plt.subplots(figsize=[10, 8])
mplhep.style.use("CMS")

for feb_id in np.unique(ak.flatten(events['feb_id'])):
    feb_events = events[feb_vars]
    feb_events = feb_events[feb_events['feb_id'] == feb_id]
    
    for amp_var in amp_var_dict:
    
        for ch in np.unique(ak.flatten(feb_events['channel_id'])):
            sel_events = (ak.sum((feb_events[amp_var] == -1) & (feb_events['channel_id'] == ch), axis=-1) > 0)
            # negative amplitude value for given channel id
            print(ch, sum(sel_events))
            if sum(sel_events) != 0:
                plt.hist(events['event_nbr_extended'][sel_events], histtype='step', linewidth=1.5, label=f"Channel {ch}")
        # plt.yscale('log')
        plt.legend()
        plt.ylabel("Entries")
        plt.xlabel("Event number")
        plt.show()
        plt.close()
# plt.savefig("Event_nbr_missing_hg.pdf")

In [ ]:
ch_nbrs = [7, 9, 20, 22, 23, 28, 29]
feb_events = events[feb_vars]
feb_events = feb_events[feb_events['feb_id'] == 0]

for ch_nbr in ch_nbrs:
    ch_events = feb_events[feb_events['channel_id'] == ch_nbr]
    ch_events_missHG = ch_events[ch_events['amplitude_hg'] == -1]
    amp_values = ak.flatten(ch_events_missHG['amplitude_lg'])
    amp_values_all = ak.flatten(ch_events['amplitude_lg'])
    plt.hist(amp_values, bins=200, range=[0, 200], histtype='step', linewidth=1.5, label="Missing HG value")
    plt.hist(amp_values_all, bins=200, range=[0, 200], histtype='step', linewidth=1.5, label="Total")
    plt.title(f"Channel {ch_nbr}")
    plt.yscale('log')
    plt.xlabel("amplitude_lg")
    plt.ylabel("Entries")
    plt.legend()
    plt.show()
    plt.close()